# 05 — Sélection des erreurs pour le fact-checking

## Objectif du notebook

Ce notebook est consacré à l'analyse qualitative des réponses incorrectes produites par **Gemini** et **OpenAI** lors de l'évaluation des questions médicales francophones de MediQAI.

Les notebooks précédents ont permis d'identifier les réponses pour lesquelles la prédiction du modèle diffère de la réponse de référence. Cependant, une réponse incorrecte ne constitue pas nécessairement une hallucination.

L'objectif de cette étape est donc de dépasser la simple mesure de l'exactitude afin d'examiner le contenu factuel des justifications générées par les modèles.

L'analyse se concentre en priorité sur des erreurs produites avec une **confiance déclarée élevée**, car ces situations correspondent à des cas où le modèle fournit une réponse incorrecte tout en exprimant une forte certitude.

Pour chaque réponse sélectionnée, l'analyse cherchera notamment à distinguer :

- une erreur de sélection de la proposition ;
- une justification médicalement incorrecte ;
- une affirmation non étayée ou inventée ;
- une contradiction avec les connaissances médicales de référence ;
- une réponse incorrecte dont le raisonnement reste néanmoins factuellement défendable.

Cette étape vise ainsi à constituer un sous-ensemble de cas pertinents destiné à une analyse qualitative et à un fact-checking manuel.

## 1. Initialisation de l'environnement

Les bibliothèques nécessaires à la manipulation des données et au futur processus de fact-checking sont importées.

Les chemins du projet sont ensuite définis afin d'accéder aux résultats de l'évaluation précédente et aux données originales de MediQAI.

In [1]:
from pathlib import Path
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv

## 2. Chargement du corpus d'erreurs

Le fichier `erreur_LLM.parquet`, produit lors de l'évaluation précédente, contient les réponses incorrectes identifiées pour Gemini et OpenAI.

Chaque observation conserve notamment :

- l'identifiant de la question ;
- le modèle évalué ;
- la réponse prédite ;
- la réponse de référence ;
- la justification produite ;
- la confiance déclarée ;
- la spécialité médicale.

Ce corpus constitue la population initiale à partir de laquelle seront sélectionnés les cas soumis au fact-checking.

In [2]:
PROJECT_ROOT = Path.cwd().parent

RESULTS_DIR = (
    PROJECT_ROOT
    / "final_evaluation"
)

RESULTS_DATA = (
    RESULTS_DIR
    / "erreur_LLM.parquet"
)

df_errors_results = pd.read_parquet(RESULTS_DATA)

In [3]:
display(df_errors_results)

,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_23245,Orthopedics,"Cas clinique :\nUn jeune homme de 25 ans, vict...",E,B,False,L'attitude vicieuse du membre inférieur droit ...,95.0
1,Gemini,mcqu_train_11747,Infectious Diseases,Cas clinique :\nVous êtes amené à voir à votre...,B,A,False,Devant une angine streptococcique (à streptoco...,95.0
2,Gemini,mcqu_train_7689,Nephro-Urology,Question :\nUne insuffisance rénale aiguë par ...,A,B,False,L'insuffisance rénale aiguë par obstacle est p...,100.0
3,Gemini,mcqu_train_18793,Pulmonology,Cas clinique :\nUn homme de 40 ans a ressenti ...,B,C,False,Chez un patient de 40 ans (ou de plus de 40 an...,90.0
4,Gemini,mcqu_train_12851,Pulmonology,"Cas clinique :\nMonsieur M., manoeuvre de son ...",C,A,False,L'image décrit une opacité en bande d'environ ...,90.0
...,...,...,...,...,...,...,...,...,...
808,OpenAI,mcqu_train_948,Cardiology,Question :\nQuelle est la dose de xylocaïne à ...,C,B,False,La xylocaïne (lidocaïne) utilisée en préventio...,88.0
809,OpenAI,mcqu_train_25805,Gynecology and Obstetrics,"Cas clinique :\nMadame F.Z. 24 ans, à déjà acc...",C,B,False,"Après un accouchement prématuré, on réalise sy...",93.0
810,OpenAI,mcqu_train_24849,Microbiology,"Question :\nEn cas de septicémie, l'antibiothé...",B,D,False,"En cas de septicémie, l’antibiothérapie doit ê...",78.0
811,OpenAI,mcqu_train_21161,Ophthalmology,Cas clinique :\nUn homme âgé de 65 ans consult...,B,A,False,Une occlusion de la veine centrale de la rétin...,78.0


## 3. Sélection des cas pour le fact-checking

Afin de construire un échantillon pertinent pour l'analyse qualitative, seules les questions répondant à plusieurs critères sont conservées.

La sélection repose sur les conditions suivantes :

- la réponse doit être incorrecte pour les deux modèles ;
- le même `sample_id` doit être disponible pour Gemini et OpenAI ;
- la confiance déclarée doit être supérieure à **90 % pour les deux modèles** ;
- les spécialités médicales doivent être diversifiées autant que possible.

Une question est ensuite sélectionnée par spécialité médicale parmi les observations répondant à ces critères.

Cette stratégie permet d'obtenir un échantillon couvrant plusieurs domaines médicaux tout en privilégiant des situations particulièrement intéressantes : **les deux modèles se trompent sur la même question tout en exprimant une confiance élevée**.

Ces observations ne sont pas considérées a priori comme des hallucinations. Elles constituent des **cas candidats au fact-checking**.

In [4]:
# Garde uniquement les réponses avec une confiance > 90
df_high_confidence = df_errors_results.loc[
    df_errors_results["confidence"] > 90
].copy()

# Garde uniquement les sample_id présents pour les deux modèles
sample_model_count = (
    df_high_confidence
    .groupby("sample_id")["model_name"]
    .nunique()
)

common_sample_ids = sample_model_count[
    sample_model_count == 2
].index

df_candidates = df_high_confidence.loc[
    df_high_confidence["sample_id"].isin(common_sample_ids)
].copy()

# Sélectionne un sample_id par spécialité médicale :
selected_samples = (
    df_candidates[
        ["sample_id", "medical_subject"]
    ]
    .drop_duplicates(subset="sample_id")
    .sort_values("medical_subject")
    .groupby("medical_subject")
    .head(1)
)

# Garde les 30 sample_id sélectionnés
selected_sample_ids = (
    selected_samples["sample_id"]
    .head(30)
)

# DataFrame final : 30 questions × 2 modèles = 60 lignes
df_selected = (
    df_candidates.loc[
        df_candidates["sample_id"].isin(
            selected_sample_ids
        )
    ]
    .sort_values(
        ["sample_id", "model_name"]
    )
    .reset_index(drop=True)
)

In [5]:
display(df_selected.head())

,model_name,sample_id,medical_subject,question_context,reference_letter,model_response,is_correct,justification,confidence
0,Gemini,mcqu_train_10195,Neurology,"Cas clinique :\nUn homme de 57 ans, ouvrier ag...",E,A,False,Le tableau clinique évoque fortement un accide...,100.0
1,OpenAI,mcqu_train_10195,Neurology,"Cas clinique :\nUn homme de 57 ans, ouvrier ag...",E,A,False,Le tableau évoque un accident vasculaire céréb...,98.0
2,Gemini,mcqu_train_14528,Hepato-Gastroenterology,"Question :\nUn ictère progressif avec prurit, ...",B,D,False,"L'association d'un ictère nu (progressif, sans...",100.0
3,OpenAI,mcqu_train_14528,Hepato-Gastroenterology,"Question :\nUn ictère progressif avec prurit, ...",B,D,False,"Un ictère progressif, indolore, avec prurit, s...",97.0
4,Gemini,mcqu_train_19766,Occupational Medicine,"Question :\nParmi les propositions suivantes, ...",D,C,False,Le certificat médical final de guérison attest...,100.0


In [6]:
#df_selected.to_csv(RESULTS_DIR / "error_selected.csv", index=False,  encoding="utf-8-sig")

## 4. Récupération des questions originales

Les `sample_id` utilisés dans le benchmark suivent une convention comprenant notamment le split et l'identifiant original de la question.

L'identifiant numérique est extrait afin de retrouver les observations correspondantes dans le fichier MCQU original de MediQAI.

Cette étape permet de revenir aux données sources avant le fact-checking et de disposer du contenu original de chaque question et de ses propositions.

In [7]:
df_id = df_selected[["sample_id"]].copy()

In [8]:
df_id["sample_id"] = (
    df_id["sample_id"]
    .str.extract(r"(\d+)$")[0]
    .astype(int)
)

In [9]:


DATA_ROOT = (
    PROJECT_ROOT
    / "data" / "raw" / "medical"
)

DATA= (
    DATA_ROOT
    / "medical_mcqu_train.parquet"
)

df_data = pd.read_parquet(DATA)

### 4.1 Vérification des données sources

Les questions correspondant aux identifiants sélectionnés sont extraites du dataset MediQAI original.

Cette récupération permet de conserver une distinction entre :

- les **données sources**, provenant directement de MediQAI ;
- les **réponses générées**, provenant de Gemini et OpenAI ;
- les futures **informations de fact-checking**, provenant des sources médicales utilisées pour vérifier les affirmations des modèles.

Cette séparation est essentielle afin de garantir la traçabilité de l'annotation.

In [10]:
df_data["id"] = df_data["id"].astype(int)
df_filtered = df_data.loc[
    df_data["id"].isin(
        df_id["sample_id"]
    )
].copy()

In [11]:
display(df_filtered.head())

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type,question_length_chars,question_length_words
185,21179,"Ce nourrisson de 2 mois, vivant dans des condi...",La soeur jumelle du nourrisson élevée par la g...,Il n'y a rien à faire,Vous isolez et séparez le frère et la soeur,Vous vaccinez la soeur,Vous prescrivez un macrolide à la soeur,Vous prescrivez un aminoside à la soeur,B,QCU,Infectious Diseases,Reasoning,208,32
455,23363,None,(cochez la réponse juste) Parmi les propositio...,1+2+3,1+2+3+4,1+3+5,2+3+5,1+3+4+5,D,QCU,Rheumatology,Understanding,316,37
1392,19916,None,Quel est le médicament dont il faudra contrôle...,Antivitamine K,Héparine,Diurétique thiazidique,Lithium,Sulfamide hypoglycémiant,A,QCU,Pharmacology,Understanding,134,20
1617,26052,None,(cochez la réponse fausse) La contraception pa...,Est indiquée dans le post partum immédiat,Est indiquée chez la femme cardiaque,Est indiquée chez la femme hypertendue,Est prise de façon discontinue 21 jours par mois,Assure une contraception par effets périphériques,C,QCU,Gynecology and Obstetrics,Understanding,76,10
1817,22488,"Un patient âgé de 45 ans, consulte pour céphal...",Le malade est mis sous traitement substitutif ...,Rhinorrhée,Diabète insipide,Sinusite maxillaire,Méningite,Apoplexie hypophysaire 12.\r\n3 mois après la ...,C,QCU,Endocrinology and Metabolism,Understanding,292,44


## Conclusion

Ce notebook a permis de constituer un sous-ensemble de réponses incorrectes particulièrement pertinentes pour la poursuite de l'analyse des hallucinations.

À partir du corpus d'erreurs obtenu lors de l'évaluation précédente, les questions pour lesquelles **Gemini et OpenAI fournissent tous deux une réponse incorrecte avec une confiance supérieure à 90 %** ont été identifiées. Une sélection par spécialité médicale a ensuite été réalisée afin de diversifier les domaines représentés dans l'échantillon.

Les questions originales correspondantes ont finalement été récupérées dans les données sources de MediQAI. Cette étape permet de disposer, pour chaque cas sélectionné, des informations nécessaires à la confrontation entre la réponse de référence du dataset et les réponses générées par les deux modèles.

Les observations sélectionnées ne sont pas considérées automatiquement comme des hallucinations. Elles constituent un **corpus de cas candidats au fact-checking**, pour lesquels une analyse du contenu médical des réponses et des justifications est nécessaire.

La suite de cette analyse est réalisée manuellement dans un fichier d'annotation dédié, permettant de confronter les réponses des modèles et les références de MediQAI à des sources médicales externes. Les résultats de cette analyse qualitative ainsi que ses limites seront présentés et discutés dans le rapport.